# Wikidata genre tree — exploration

Exploratory notebook over the Bronze `wikidata_genre_tree.parquet` output (see [`../SCHEMA.md`](../SCHEMA.md#bronze)) and the Silver `1_genre_classification`/`2_regional_classification`/`3_genre_parents`/`4_hierarchy` outputs (see [`../SCHEMA.md`](../SCHEMA.md#silver)). Reads local Parquet files only — no live SPARQL calls.

**Prerequisite:** run the ingest and silver steps first so the Parquet files exist:

```bash
uv run --package wikidata python -m wikidata.ingest
uv run --package wikidata python -m wikidata.silver
```

## P31 vs. P279/P361 — how the genre set and the hierarchy are built

Two separate questions, two separate properties (see [`../SCHEMA.md#wikidata-properties-used`](../SCHEMA.md#wikidata-properties-used)):

- **"What is the full set of genres?"** → `P31` ("instance of"). `wd:Q11399` "rock music" `wdt:P31` `wd:Q188451` "music genre" — this class-membership edge is how Bronze finds all ~6,300 genre items (`GENRE_TREE_QUERY`'s `?item wdt:P31 wd:Q188451`).
- **"How do two genres relate?"** → `P279` ("subclass of") or `P361` ("part of"), *between genre items*. `wd:Q11399` "rock music" `wdt:P279` `wd:Q373342` "popular music" is the hierarchy edge.

`P31` is **never** used to build hierarchy edges, and `P279`/`P361` are **never** used to find the genre set — a `P279*` walk from `Q188451` itself only reaches ~14 meta-category items ("rock genre", "jazz genre", ...), not real genres, because a genre's `P279` parent chain leads to broader *concepts* like "popular music", not back to the literal `Q188451` node. Confirmed live: `ASK { wd:Q373342 wdt:P279* wd:Q188451 }` → `false`, even though "popular music" is itself `P31`-classified as a music genre.

In [ ]:
import networkx as nx
import polars as pl
from common.env import load_pipeline_env, require_env, resolve_pipeline_path

import wikidata

load_pipeline_env(wikidata.__file__)

bronze_output_dir = resolve_pipeline_path(wikidata.__file__, require_env("BRONZE_OUTPUT_DIR"))
bronze_path = bronze_output_dir / "wikidata_genre_tree.parquet"
df = pl.read_parquet(bronze_path)
df.shape

## Tabular exploration

In [ ]:
df.head(10)

In [ ]:
df.group_by("relation_type").len().sort("len", descending=True)

In [ ]:
roots = df.filter(pl.col("parent_id").is_null())
print(f"{roots.height} root items")
roots.select("item_id", "item_label").head(10)

In [ ]:
multi_parent = (
    df.filter(pl.col("parent_id").is_not_null())
    .group_by("item_id", "item_label")
    .agg(pl.col("parent_id").n_unique().alias("n_parents"))
    .filter(pl.col("n_parents") > 1)
    .sort("n_parents", descending=True)
)
print(f"{multi_parent.height} items with more than one parent")
multi_parent.head(10)

## Graph visualization

The full tree (~9,700 edges) is too dense to render usefully in one plot, so build the full graph for structural stats, then plot just the neighborhood around a single genre.

In [ ]:
G = nx.DiGraph()
for row in df.iter_rows(named=True):
    G.add_node(row["item_id"], label=row["item_label"])
    if row["parent_id"] is not None:
        G.add_edge(row["item_id"], row["parent_id"], relation=row["relation_type"])

print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"{nx.number_weakly_connected_components(G)} weakly connected components")

In [ ]:
import matplotlib.pyplot as plt

ITEM_ID = "Q11399"  # rock music — change this to explore a different genre
RADIUS = 1  # networkx's spring_layout needs scipy for subgraphs of 500+ nodes; radius=1 stays well under that

undirected = G.to_undirected()
subgraph = nx.ego_graph(undirected, ITEM_ID, radius=RADIUS)


def edge_relation(u: str, v: str) -> str | None:
    return G.get_edge_data(u, v, {}).get("relation") or G.get_edge_data(v, u, {}).get("relation")


edge_colors = ["tab:blue" if edge_relation(u, v) == "P279" else "tab:orange" for u, v in subgraph.edges()]
labels = {n: G.nodes[n].get("label", n) for n in subgraph.nodes()}

plt.figure(figsize=(12, 10))
pos = nx.spring_layout(subgraph, seed=42)
nx.draw_networkx_nodes(subgraph, pos, node_size=300, node_color="lightgray")
nx.draw_networkx_edges(subgraph, pos, edge_color=edge_colors, alpha=0.6)
nx.draw_networkx_labels(subgraph, pos, labels=labels, font_size=8)
plt.title(f"Genre tree neighborhood: {labels[ITEM_ID]} (radius={RADIUS})\nblue = P279, orange = P361")
plt.axis("off")
plt.show()

## Silver exploration

`1_genre_classification.parquet` carries the first Silver columns — `is_genre`/`classification_reason` — on top of the unchanged Bronze edge list.

In [ ]:
silver_output_dir = resolve_pipeline_path(wikidata.__file__, require_env("SILVER_OUTPUT_DIR"))

genre_class_df = pl.read_parquet(silver_output_dir / "1_genre_classification.parquet")
genre_class_df.shape

In [ ]:
genre_class_df.group_by("is_genre", "classification_reason").len().sort("len", descending=True)

In [ ]:
tagged = genre_class_df.filter(~pl.col("is_genre")).select("item_id", "item_label", "classification_reason").unique()
print(f"{tagged.height} distinct items tagged non-genre (not excluded — see 2_regional_classification below)")
tagged.head(10)

### 2_regional_classification — regional genre cascade

`2_regional_classification.parquet` adds `is_regional`/`regional_reason`, cascading a "regional" flag down from `1_genre_classification`'s `regional_overview` seeds (e.g. "music of Cape Verde") to every descendant genre item, via an **any-parent** rule: an item is regional if *any* one of its parent edges points to a seed or an already-regional item (see [`../SCHEMA.md#2_regional_classification`](../SCHEMA.md#2_regional_classification)). The seed items themselves are flagged `is_regional = true` / `regional_reason = "seed"` — they are regional genre nodes in their own right, not merely a launching point for other items. Flag-only, no rows dropped.

**Exploration-phase finding**: on a real run this flags ~53% of all items (299 seed, 1,392 direct, 1,686 inherited), driven by continent-level seeds ("music of Asia", "music of Europe", ...) with large fan-out — kept as-is for now rather than narrowed, since the design is still exploratory.

In [ ]:
regional_df = pl.read_parquet(silver_output_dir / "2_regional_classification.parquet")
regional_df.shape

In [ ]:
regional_df.group_by("is_regional", "regional_reason").len().sort("len", descending=True)

In [ ]:
# music of Cape Verde is the seed itself; morna's only parent is that seed (direct); fado's
# parent is morna (inherited).
regional_df.filter(pl.col("item_label").is_in(["music of Cape Verde", "morna", "fado"])).select(
    "item_id", "item_label", "parent_id", "parent_label", "is_regional", "regional_reason"
).unique()

### 3_genre_parents

`3_genre_parents.parquet` adds `parent_is_genre`, flagging whether each edge's `parent_id` is itself flagged `is_genre = True` by `1_genre_classification` (see [`../SCHEMA.md#3_genre_parents`](../SCHEMA.md#3_genre_parents)). Flag-only, no rows dropped.

In [ ]:
genre_parents_df = pl.read_parquet(silver_output_dir / "3_genre_parents.parquet")
genre_parents_df.shape

In [ ]:
genre_parents_df.group_by("parent_is_genre").len().sort("len", descending=True)

In [ ]:
non_genre_parents = (
    genre_parents_df.filter(~pl.col("parent_is_genre"))
    .select("item_id", "item_label", "parent_id", "parent_label", "relation_type")
    .unique()
)
print(f"{non_genre_parents.height} edges into a non-genre parent")
non_genre_parents.head(10)

In [ ]:
canonical_genres_roots = genre_parents_df.filter(pl.col("parent_id").is_null() & ~pl.col("is_regional")).select(
    "item_id", "item_label"
)
print(f"{canonical_genres_roots.height} canonical roots")
print(canonical_genres_roots)

### 4_hierarchy — pruned, single-parent-per-item (canonical)

The first Silver step that drops rows: filters to genre-only, non-regional edges, then collapses any item with more than one surviving parent down to the one with the lowest numeric QID — a **provisional heuristic** (see [`../SCHEMA.md#4_hierarchy`](../SCHEMA.md#4_hierarchy)), not a considered rule. This is the canonical output (`4_hierarchy.parquet`); regional items (`is_regional = true`) are excluded here and land in `4_regional_hierarchy.parquet` instead — see the section below.

In [ ]:
hierarchy_df = pl.read_parquet(silver_output_dir / "4_hierarchy.parquet")
hierarchy_df.shape

In [ ]:
genre_items = genre_parents_df.filter(pl.col("is_genre")).select("item_id").unique()
surviving_items = hierarchy_df.select("item_id").unique()
missing_from_canonical = genre_items.join(surviving_items, on="item_id", how="anti")
print(
    f"{missing_from_canonical.height} of {genre_items.height} genre items have zero rows in 4_hierarchy "
    "(either regional, routed to 4_regional_hierarchy instead, or every parent edge was non-genre)"
)
missing_from_canonical.head(10)

In [ ]:
# Sample of items with more than one genre-only parent candidate, and which one the lowest-QID
# heuristic kept (chosen_parent_id) vs. the alternatives it discarded.
multi_parent_candidates = genre_parents_df.filter(pl.col("is_genre") & pl.col("parent_is_genre")).select(
    "item_id", "item_label", "parent_id", "parent_label"
)
multi_parent_items = multi_parent_candidates.group_by("item_id").len().filter(pl.col("len") > 1).select("item_id")
print(f"{multi_parent_items.height} items have more than one genre parent")

sample = (
    multi_parent_candidates.join(multi_parent_items, on="item_id")
    .join(hierarchy_df.select("item_id", pl.col("parent_id").alias("chosen_parent_id")), on="item_id")
    .sort("item_id")
)
sample.head(10)

### Hierarchy graph visualization

`4_hierarchy` is a genre-only forest — unlike the Bronze graph above, each item has at most one parent — so the same neighborhood plot shows what the lowest-QID collapse actually kept.

In [ ]:
H = nx.DiGraph()
for row in hierarchy_df.iter_rows(named=True):
    H.add_node(row["item_id"], label=row["item_label"])
    if row["parent_id"] is not None:
        H.add_edge(row["item_id"], row["parent_id"], relation=row["relation_type"])

print(f"{H.number_of_nodes()} nodes, {H.number_of_edges()} edges")
print(f"{nx.number_weakly_connected_components(H)} weakly connected components")

In [ ]:
h_undirected = H.to_undirected()
h_subgraph = nx.ego_graph(h_undirected, ITEM_ID, radius=RADIUS)
h_labels = {n: H.nodes[n].get("label", n) for n in h_subgraph.nodes()}

plt.figure(figsize=(12, 10))
h_pos = nx.spring_layout(h_subgraph, seed=42)
nx.draw_networkx_nodes(h_subgraph, h_pos, node_size=300, node_color="lightgray")
nx.draw_networkx_edges(h_subgraph, h_pos, alpha=0.6)
nx.draw_networkx_labels(h_subgraph, h_pos, labels=h_labels, font_size=8)
plt.title(f"4_hierarchy neighborhood: {h_labels[ITEM_ID]} (radius={RADIUS})")
plt.axis("off")
plt.show()

### 4_regional_hierarchy — pruned, single-parent-per-item (regional)

`4_regional_hierarchy.parquet` mirrors `4_hierarchy`'s pruning, restricted to `is_regional = true` items — which now includes the `regional_overview` seeds themselves as real nodes with real parent chains (or genuine roots, if a seed has no `P279`/`P361` parent of its own), rather than being dropped or promoted. An item like "morna," whose only parent is the "music of Cape Verde" seed, keeps that real parent edge instead of being promoted to a synthetic root (see [`../SCHEMA.md#4_hierarchy`](../SCHEMA.md#4_hierarchy)).

In [ ]:
regional_hierarchy_df = pl.read_parquet(silver_output_dir / "4_regional_hierarchy.parquet")
regional_hierarchy_df.shape

In [ ]:
# Every root here should be a regional_overview seed (regional_reason == "seed") — unlike the
# earlier design, non-seed regional items are never promoted to a synthetic root, since seeds
# are themselves real nodes with real parent chains.
roots = regional_hierarchy_df.filter(pl.col("parent_id").is_null()).select("item_id", "item_label")
root_reasons = roots.join(regional_df.select("item_id", "regional_reason").unique(), on="item_id", how="left")
print(f"{root_reasons.height} roots in 4_regional_hierarchy")
root_reasons.group_by("regional_reason").len().sort("len", descending=True)

In [ ]:
R = nx.DiGraph()
for row in regional_hierarchy_df.iter_rows(named=True):
    R.add_node(row["item_id"], label=row["item_label"])
    if row["parent_id"] is not None:
        R.add_edge(row["item_id"], row["parent_id"], relation=row["relation_type"])

print(f"{R.number_of_nodes()} nodes, {R.number_of_edges()} edges")
print(f"{nx.number_weakly_connected_components(R)} weakly connected components")

In [ ]:
REGIONAL_ITEM_ID = "Q185676"  # fado — rock music won't appear in this graph, so use a regional example instead

r_undirected = R.to_undirected()
r_subgraph = nx.ego_graph(r_undirected, REGIONAL_ITEM_ID, radius=RADIUS)
r_labels = {n: R.nodes[n].get("label", n) for n in r_subgraph.nodes()}

plt.figure(figsize=(12, 10))
r_pos = nx.spring_layout(r_subgraph, seed=42)
nx.draw_networkx_nodes(r_subgraph, r_pos, node_size=300, node_color="lightgray")
nx.draw_networkx_edges(r_subgraph, r_pos, alpha=0.6)
nx.draw_networkx_labels(r_subgraph, r_pos, labels=r_labels, font_size=8)
plt.title(f"4_regional_hierarchy neighborhood: {r_labels[REGIONAL_ITEM_ID]} (radius={RADIUS})")
plt.axis("off")
plt.show()